# 00 — Master notebook (run everything)

End-to-end workflow without CLI. Includes a token-cost-controlled preview mode (`LIMIT_SECTIONS`) and a monitoring section that verifies chunks are produced at the expected ~8K token size.

**Prereqs**
- `pip install -r ../requirements.txt`
- `.env` linked
- Qdrant container up (`docker compose -f ../docker/docker-compose.yml up -d`)
- Dashboard: http://localhost:6333/dashboard

## 1. Bootstrap

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.core.config import settings
from src.core.qdrant_store import client
qc = client()
print('Qdrant:', settings.qdrant_host, settings.qdrant_port)
print('Collections:', [c.name for c in qc.get_collections().collections])
print('Embedding:', settings.embedding_model)
print('Default LLM:', settings.default_provider)

## 2. Status — what is already ingested

In [ ]:
from src.ingestion import manifest
rows = manifest.list_books()
if not rows: print('Manifest empty.')
for r in rows:
    print(f"- {r['book']} / {r['chapter']}  chunks={r['chunks']}  provider={r['provider']}  at={r['ingested_at']}")

## 3. Ingest — knobs

`LIMIT_SECTIONS`: ingest only the first N sections. Use a small number (1-3) while iterating on prompts/chunking to keep API cost low. Set to `None` for full chapter.

When `LIMIT_SECTIONS != None` the manifest is NOT written — this is a preview run.

In [ ]:
BOOK = 'islp'
CHAPTER = 'ch02'
PROVIDER = 'openai'        # or 'deepseek'
FORCE = True
LIMIT_SECTIONS = 1          # set None for full chapter

from src.ingestion.pipeline import run_chapter
result = run_chapter(BOOK, CHAPTER, provider=PROVIDER, force=FORCE, limit_sections=LIMIT_SECTIONS)
result

## 4. Monitoring — token distribution

Asserts:
1. No chunk above the embedding token cap (`oversize == 0`).
2. Sections above 8000 tokens are split.
3. Token histogram shows distribution across <1k / 1-2k / 2-4k / 4-6k / 6-8k / >8k buckets.

In [ ]:
import json
from pathlib import Path
stats = json.loads(Path(f'../data/parsed/{BOOK}/{CHAPTER}_build_stats.json').read_text())
print(f"sections={stats['n_sections']}  chunks={stats['n_chunks']}  split={stats['split_sections']}  oversize={stats['n_oversize']}")
print(f"tokens min/mean/max = {stats['token_min']} / {stats['token_mean']:.0f} / {stats['token_max']}")
print('histogram:', stats['token_histogram'])
assert stats['n_oversize'] == 0, 'Found chunks above the embedding token cap'
print('\nper-section breakdown:')
for s in stats['per_section'][:10]:
    print(f"  {s['section_id'][:50]:50s} chunks={s['n_chunks']} tokens={s['tokens']}")

## 5. Verify in Qdrant — pull payloads, recompute tokens

In [ ]:
from src.ingestion.build_documents import count_tokens
tx = settings.qdrant_collection_text
print(f'{tx}: {qc.count(tx).count} points')
res, _ = qc.scroll(tx, limit=20, with_payload=True, with_vectors=False)
for p in res:
    pl = p.payload
    actual = count_tokens(pl.get('text',''))
    stored = pl.get('token_count')
    ok = 'OK' if actual == stored else 'MISMATCH'
    print(f"[{ok}] h1={pl.get('h1','?')[:24]:24s}  h2={pl.get('h2_path','?')[:46]:46s}  pages={pl.get('page_from')}-{pl.get('page_to')}  tokens={actual}")

## 6. Inspect one full payload — confirm metadata schema

In [ ]:
res, _ = qc.scroll(tx, limit=1, with_payload=True, with_vectors=False)
pl = res[0].payload
for k in ['book_slug','book_name','authors','chapter_id','section_id','h1','h2_path',
         'page_from','page_to','token_count','chunk_index','n_chunks_in_section',
         'has_formula','has_image','has_table','n_formulas','synopsis','index_extended']:
    v = pl.get(k)
    if isinstance(v, str) and len(v) > 200: v = v[:200] + '...'
    print(f'  {k:25s} = {v!r}')

## 7. Text RAG query

In [ ]:
QUESTION = 'Explain the bias-variance tradeoff in supervised learning.'
BOOK_FILTER = BOOK
from src.services.retrieval.chain import build_chain
chain = build_chain(book_slug=BOOK_FILTER)
r = chain.invoke(QUESTION)
print('=== ANSWER ===\n')
print(r['answer'])
print('\n=== SOURCES ===')
for i, d in enumerate(r['sources'], 1):
    m = d.metadata
    print(f"[{i}] {m.get('book_name','?')[:40]} / {m.get('h2_path','?')[:50]} pp.{m.get('page_from')}-{m.get('page_to')}")
    print('   ', d.page_content[:200].replace('\n',' '), '...')

## 8. Image search by caption

In [ ]:
IMG_QUERY = 'wage vs age figure'
from src.services.retrieval.retrievers import search_images
hits = search_images(IMG_QUERY, book_slug=BOOK_FILTER, k=5)
for i, h in enumerate(hits, 1):
    print(f"[{i}] score={h['score']:.3f}  {h['image_name']}  page={h.get('page')}")
    print(f"     ref: {h['image_reference'][:140]}")

## 9. Preview top image inline

In [ ]:
from pathlib import Path
from IPython.display import Image, display
if hits:
    p = hits[0]['image_path']
    display(Image(filename=p, width=520)) if Path(p).exists() else print('Not on disk:', p)
else:
    print('No image hits.')

## 10. Wipe collections + reset LLM cache (DANGEROUS)

In [ ]:
# from src.core.qdrant_store import ensure_text_collection, ensure_image_collection
# import shutil
# qc.delete_collection(settings.qdrant_collection_text)
# qc.delete_collection(settings.qdrant_collection_images)
# ensure_text_collection(); ensure_image_collection()
# shutil.rmtree(f'../data/parsed/{BOOK}/cache', ignore_errors=True)
print('Uncomment to wipe collections + LLM cache.')

## Companion notebooks

- `02_ingest_islp_ch2.ipynb` — stage-by-stage walkthrough
- `03_test_retrieval.ipynb` — retrieval + metadata filters
- `04_compare_providers.ipynb` — OpenAI vs DeepSeek
- `05_inspect_qdrant.ipynb` — Qdrant DB tour